# Project: Matrix Completion Using Stochastic Gradient Descent
### Soren Patel

**AI Usage Statement**  
I used AI in this lab for reading/cleaning the data file into 3-tuples and creating the bidirectional index mapping dictionaries. I also used it for part of my results analysis, in determining the big O time complexity of the algorithm and the military applications. Lastly, I used it to collapse my already written algorithm into the helper method at the very end for further experimentation.

## 1. Load, Clean, and Format Data  
Load data from the given CSV file and format all data points as a 3-tuples: (user_idx, movie_idx, rating)

Import necessary libraries

In [30]:
import pandas as pd
import numpy as np

Read in the file and clean data, keeping users with 20+ ratings and movies with 10+ ratings. This is to ensure that the computed averages for matrix completion in the future are accurately representative of the dataset as a whole and not skewed by one errant point.

In [31]:
ratings_df = pd.read_csv("ml-latest-small/ratings.csv")

user_counts = ratings_df['userId'].value_counts()
movie_counts = ratings_df['movieId'].value_counts()

valid_users = user_counts[user_counts >= 20].index
valid_movies = movie_counts[movie_counts >= 10].index

df_filtered = ratings_df[
    ratings_df['userId'].isin(valid_users) & 
    ratings_df['movieId'].isin(valid_movies)
].copy()

Create zero-based index mappings for all users and movies, so they are accessible as array indices in future matrices P and Q

In [32]:
unique_users = df_filtered['userId'].unique()
unique_movies = df_filtered['movieId'].unique()

user2idx = {raw_id: idx for idx, raw_id in enumerate(unique_users)}
movie2idx = {raw_id: idx for idx, raw_id in enumerate(unique_movies)}

idx2user = {idx: raw_id for raw_id, idx in user2idx.items()}
idx2movie = {idx: raw_id for raw_id, idx in movie2idx.items()}

Format into a list of tuples: (user_idx, movie_idx, rating)

In [33]:
ratings_tuples = [
    (user2idx[u], movie2idx[m], float(r))
    for u, m, r in zip(df_filtered['userId'], df_filtered['movieId'], df_filtered['rating'])
]

Print dataset metadata as confirmation that load/cleaning processes worked.

In [34]:
print(f"Total ratings loaded: {len(ratings_tuples)}")
print(f"Total Unique Users: {len(user2idx)}")
print(f"Total Unique Movies: {len(movie2idx)}")
print(f"Sample observation tuple: {ratings_tuples[0]}")

Total ratings loaded: 81116
Total Unique Users: 610
Total Unique Movies: 2269
Sample observation tuple: (0, 0, 4.0)


## 2. Insert Personal Ratings  
Define and insert personal ratings of 20 movies. The eventual recommendation algorithm will be based on these, recommending the top and bottom 10 movies for a person who's taste reflects these.

Create a list of raw ratings and a new user index. Append each one to the list of rating tuples in the proper format. Include a warning if a brand new movie which was not previously in the dataset was added, and print total ratings at the end for confirmation.

In [35]:
movies_df = pd.read_csv("ml-latest-small/movies.csv")
title2id = movies_df.set_index('title')['movieId'].to_dict()

my_raw_ratings = {
    "Toy Story (1995)": 5.0,
    "Star Wars: Episode IV - A New Hope (1977)": 5.0,
    "Matrix, The (1999)": 5.0,
    "Pulp Fiction (1994)": 4.5,
    "Dark Knight, The (2008)": 5.0,
    "Terminator 2: Judgment Day (1991)": 5.0,
    "Die Hard (1988)": 5.0,
    "Superbad (2007)": 4.5,
    "Hangover, The (2009)": 4.5,
    "Back to the Future (1985)": 4.5,
    "Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)": 5.0,
    "Anchorman: The Legend of Ron Burgundy (2004)": 4.0,
    "Gladiator (2000)": 4.5,
    "Jurassic Park (1993)": 2.0,
    "Independence Day (a.k.a. ID4) (1996)": 1.0,
    "Titanic (1997)": 1.0,
    "Notebook, The (2004)": 1.0,
    "Sleepless in Seattle (1993)": 1.5,
    "Pretty Woman (1990)": 1.5,
    "Ghost (1990)": 2.0,
}

MY_USER_IDX = len(user2idx)

user2idx["MY_CUSTOM_USER"] = MY_USER_IDX
idx2user[MY_USER_IDX] = "MY_CUSTOM_USER"

added_count = 0

for title, rating in my_raw_ratings.items():
    raw_id = title2id.get(title)
    
    if raw_id in movie2idx:
        m_idx = movie2idx[raw_id]
        ratings_tuples.append((MY_USER_IDX, m_idx, float(rating)))
        added_count += 1
    else:
        print(f"Warning: '{title}' was not found in filtered dataset.")

print(f"Successfully added User ID {MY_USER_IDX} with {added_count} ratings.")
print(f"New total ratings in dataset: {len(ratings_tuples)}")

Successfully added User ID 610 with 20 ratings.
New total ratings in dataset: 81136


## 3. Split Data Into Training and Testing Sets  
80% training, 20% testing data

Create a copy of the rating tuples and shuffle them randomly. 2028 seed to numpy random for reproducible results. Split at 0.8*length of the data, creating an 80%/20% split. Print training and testing set sizes for confirmation.

In [36]:
np.random.seed(2028)
all_data = ratings_tuples.copy()
np.random.shuffle(all_data)

split_idx = int(0.8 * len(all_data))
train_data = all_data[:split_idx]
test_data = all_data[split_idx:]

print(f"Total observations: {len(all_data)}")
print(f"Training set size: {len(train_data)} ({len(train_data)/len(all_data):.1%})")
print(f"Testing set size: {len(test_data)} ({len(test_data)/len(all_data):.1%})")

Total observations: 81136
Training set size: 64908 (80.0%)
Testing set size: 16228 (20.0%)


## 4. Decomposition and SGD for RMSE Minimization
Broken down into steps, we construct P and Q, calculate RMSE across epochs, and minimize RMSE using Stochastic Gradient Descent

Define hyperparamaters (rank, learning rate, and #epochs). Construct P and Q with NumPy normal distribution, scaled so initial values are around an average rating of 3.5. Create a function to calculate RMSE given the training dataset, P, and Q matrices. Apply SGD over all epochs to minimize RMSE, comparing each new RMSE value to a stored best (lowest). The algorithm cuts off when the currently calculated RMSE is greater than the previous best, indicating that we have started to "climb out" of the local minimum of error. Print RMSE values for the first epoch and every 5. After multiple tests, the SGD bottoms out at 15 epochs.  

I decided to start by defining K as 20, estimating 20 latent features that viewers would characterize movies. I also defined the learning rate as 0.05 due to it being a common value in class notes. Lastly, I set the number of epochs to 30, but as seen in the output, the SGD bottomed at 15.

In [37]:
K = 20
alpha = 0.05
epochs = 30

num_users = len(user2idx)
num_movies = len(movie2idx)

scale = np.sqrt(3.5 / K)
P = np.random.normal(loc=0.0, scale=scale, size=(num_users, K))
Q = np.random.normal(loc=0.0, scale=scale, size=(num_movies, K))

def compute_rmse(T_set, P_mat, Q_mat):
    T_size = len(T_set)
    if T_size == 0:
        return 0.0
    
    squared_errors_sum = 0.0
    for u, m, r in T_set:
        pred = np.dot(P_mat[u], Q_mat[m])
        squared_errors_sum += (r - pred) ** 2
        
    return np.sqrt((1.0 / T_size) * squared_errors_sum)

print(f"Matrix Completion | Rank K = {K} | Alpha = {alpha}")
print("-" * 65)

best_test_rmse = float('inf')

for epoch in range(1, epochs + 1):
    np.random.shuffle(train_data)
    
    for u, m, r in train_data:
        err = r - np.dot(P[u], Q[m])
        
        P_u_old = P[u].copy()
        
        P[u] += alpha * err * Q[m]
        Q[m] += alpha * err * P_u_old
    
    if epoch % 5 == 0 or epoch == 1:
        train_rmse = compute_rmse(train_data, P, Q)
        test_rmse = compute_rmse(test_data, P, Q)
        
        print(f"Epoch {epoch:02d}/{epochs} | Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
        
        if test_rmse < best_test_rmse:
            best_test_rmse = test_rmse
        elif test_rmse > best_test_rmse + 0.005 and epoch > 10:
            print(f"\nTesting error bottomed out at epoch {epoch}.")
            break

Matrix Completion | Rank K = 20 | Alpha = 0.05
-----------------------------------------------------------------
Epoch 01/30 | Train RMSE: 0.9042 | Test RMSE: 1.0700
Epoch 05/30 | Train RMSE: 0.6568 | Test RMSE: 1.0108
Epoch 10/30 | Train RMSE: 0.5607 | Test RMSE: 1.0472
Epoch 15/30 | Train RMSE: 0.5048 | Test RMSE: 1.0838

Testing error bottomed out at epoch 15.


## 5. Recommmend Top 10 and Bottom 10 Movies  
Multiply personal tastes against every other movie (removing seen ones) to predict how much each one would be liked according to my personal rankings. Sort all remaining unseen movies from highest to lowerst, and use the previously defined dictionaries to print out the titles of the top and bottom rated 10.


In [38]:
my_predictions = np.dot(P[MY_USER_IDX], Q.T)

seen_movie_indices = {m_idx for u_idx, m_idx, r in ratings_tuples if u_idx == MY_USER_IDX}

movies_df = pd.read_csv("ml-latest-small/movies.csv")
movie_id_to_title = movies_df.set_index('movieId')['title'].to_dict()

unseen_movie_predictions = []
for m_idx in range(num_movies):
    if m_idx not in seen_movie_indices:
        pred_score = my_predictions[m_idx]
        unseen_movie_predictions.append((m_idx, pred_score))

unseen_movie_predictions.sort(key=lambda item: item[1], reverse=True)

print(f"TOP 10 MOVIE PICKS FOR USER {MY_USER_IDX}")
print("=" * 70)

for rank, (m_idx, score) in enumerate(unseen_movie_predictions[:10], start=1):
    raw_movie_id = idx2movie[m_idx]
    title = movie_id_to_title.get(raw_movie_id, "Unknown Title")
    print(f"{rank:2d}. {title}")


print(f"\nBOTTOM 10 MOVIES FOR USER {MY_USER_IDX}")
print("=" * 70)

for rank, (m_idx, score) in enumerate(unseen_movie_predictions[-10:], start=1):
    raw_movie_id = idx2movie[m_idx]
    title = movie_id_to_title.get(raw_movie_id, "Unknown Title")
    print(f"{rank:2d}. {title}")

TOP 10 MOVIE PICKS FOR USER 610
 1. Lord of the Rings: The Return of the King, The (2003)
 2. Thin Blue Line, The (1988)
 3. Amazing Spider-Man, The (2012)
 4. Legend of Drunken Master, The (Jui kuen II) (1994)
 5. Drunken Master (Jui kuen) (1978)
 6. Solaris (Solyaris) (1972)
 7. X-Men: First Class (2011)
 8. Brazil (1985)
 9. Blazing Saddles (1974)
10. Descent, The (2005)

BOTTOM 10 MOVIES FOR USER 610
 1. 15 Minutes (2001)
 2. Dumb and Dumberer: When Harry Met Lloyd (2003)
 3. Days of Thunder (1990)
 4. Universal Soldier (1992)
 5. Mr. & Mrs. Smith (2005)
 6. Muppets, The (2011)
 7. Showtime (2002)
 8. Bachelor, The (1999)
 9. Expendables, The (2010)
10. Home Alone 2: Lost in New York (1992)


## 6. Analysis of Immediate Results  
My personal ratings highly favored movies that align with sci-fi, fantasy, epic stories, action, and comedy. They also lowly favored movies with dramas, romanticism, and cheesiness. Comparing the top 10 predictions to those genres, I can group the results. 
Themes found in the top 10:  
1,3,7 - epic sci/fi, fantasy, superhero/action  
4,5 - action and combat  
8,9 - comedy, sci/fi X comedy  
2,6,10 - thought provoking, epic storytelling  

For the bottom 10, analyze in a similar manner. Themes found here are not found highly in my personal preferences.   
5,8 - romance  
1,2,3,4,9,10 - mix of cheesy movies and generally low-rated films  
6 - family-oriented comedy over sharper/more mature-rated humor  
7 - seemingly an outlier.  


## 7. Further Thoughts on Application
I would use this completed matrix for a recommendation system with the following steps as a skeleton for implementation.  
1. Given a new user requesting recommendations, I would get their predictions and dot it with Qt (the movie matrix), to get the completed user rating predictions across all movies.
2. Filter out movies they had already watched
3. Rank from highest to lowest preference score and display the top 20 or so, based on the platform. For example, Netflix as the user keeps scrolling, keep loading more movies down the list.  

This approach is unique because it captures underlying themes automatically. The rank chosen represents the amount of latent factors for which the items in the matrix are being categorized, in this case movies. Other categorization and filtering algorithms, like K Nearest Neighbors ranks based on pure distance and overlap, but this method captures themes through latent compression.  

It also can handle realistic datasets. Other algorithms require data matrices to be reformed into a Compressed Sparse Row matrix, but this approach can handle the sparseness on its own. Real datasets are often 99% empty, so this is more practical. The SGD, once implemented, goes only observed entries and eventually works to predict the missing entries.

This approach also boasts strengths in its time complexity. Calculating a single recommendation score during inference requires only a vector dot product between P_u and Q_i (K multiplications), resulting in an O(1) constant inference speed. Overall, SGD wins in comparison with other methods of minimizing RMSE, with the following time complexities.  
  
Stochastic Gradient Descent: O(P * r) per epoch  
Alternating Least Squares:   O(P * r^2 + (d + n) * r^3) per iteration  
Coordinate Descent(CCD++):   O(P * r) per inner loop only  
Proximal Gradient(Nuclear):  O(d * n * r)

## 8. Defense and Military Application  
Low rank matrix completion can be used with any sparse matrix where we need to predict or infer the missing entries. Here are some examples in the military where this may be applicable.  
1. Smart billet matching: Match Sailors/Marines to duty stations or specialized training pipelines. Rows are personnel, columns are specialized billets or qualifications, and known entries are past billet performance/evaluations.
2. Targeted training recommendations: Recommend specific PQS (Personnel Qualification Standards), professional military education (PME), or certifications based on career tracks of peers with similar operational profiles
3. Parts failure profiling: Fleet maintenance platforms track thousands of parts across hundreds of ships/aircraft. If a specific class of vessel deployed in the Indo-Pacific shows a pattern of component degradation, matrix completion can predict unobserved component failure risks across other ships in similar operational environments, optimizing forward-deployed spare parts inventories before failures occur.
4. RF spectrum unobserved area mapping: Electromagnetic spectrum mapping often yields sparse sensor reads due to jamming, terrain shielding, or limited sensor coverage. Rows represent time/location slices, and columns represent frequency bands. Matrix completion reconstructs missing RF signal power levels across unobserved frequencies to build a complete tactical Operating Environment Picture.

## 9. Experimentation with My Implementation
The example shown previously yielded sensible results. But what if we test with a different k, learning rate, or epoch#? How does changing the hyperparameters affect our results? To efficiently do so, I collapsed the above code into a helper function below.

In [39]:
def run_matrix_completion_pipeline(df_filtered, my_raw_ratings, K=20, alpha=0.005, epochs=35):
    # -------------------------------------------------------------------------
    # 1. Map Raw IDs to Contiguous 0-Based Indices
    # -------------------------------------------------------------------------
    unique_users = df_filtered['userId'].unique()
    unique_movies = df_filtered['movieId'].unique()

    user2idx = {raw_id: idx for idx, raw_id in enumerate(unique_users)}
    movie2idx = {raw_id: idx for idx, raw_id in enumerate(unique_movies)}
    idx2movie = {idx: raw_id for raw_id, idx in movie2idx.items()}

    # Assign a custom user index for yourself
    MY_USER_IDX = len(user2idx)
    num_users = MY_USER_IDX + 1
    num_movies = len(unique_movies)

    # Title lookup mapping
    movies_df = pd.read_csv("ml-latest-small/movies.csv")
    title_to_id = movies_df.set_index('title')['movieId'].to_dict()
    id_to_title = movies_df.set_index('movieId')['title'].to_dict()

    # -------------------------------------------------------------------------
    # 2. Build Ratings Tuples & Append Custom User Ratings
    # -------------------------------------------------------------------------
    ratings_tuples = [
        (user2idx[u], movie2idx[m], float(r))
        for u, m, r in zip(df_filtered['userId'], df_filtered['movieId'], df_filtered['rating'])
    ]

    for title, rating in my_raw_ratings.items():
        if title in title_to_id:
            raw_m_id = title_to_id[title]
            if raw_m_id in movie2idx:
                m_idx = movie2idx[raw_m_id]
                ratings_tuples.append((MY_USER_IDX, m_idx, float(rating)))

    # Train/Test Split (80/20)
    np.random.seed(42)
    shuffled_tuples = ratings_tuples.copy()
    np.random.shuffle(shuffled_tuples)

    split_idx = int(0.8 * len(shuffled_tuples))
    train_data = shuffled_tuples[:split_idx]
    test_data = shuffled_tuples[split_idx:]

    # -------------------------------------------------------------------------
    # 3. Helper Evaluation Function
    # -------------------------------------------------------------------------
    def compute_rmse(T_set, P_mat, Q_mat):
        if len(T_set) == 0:
            return 0.0
        squared_errors = sum((r - np.dot(P_mat[u], Q_mat[m])) ** 2 for u, m, r in T_set)
        return np.sqrt(squared_errors / len(T_set))

    # -------------------------------------------------------------------------
    # 4. Initialize P and Q
    # -------------------------------------------------------------------------
    scale = np.sqrt(3.5 / K)
    P = np.random.normal(loc=0.0, scale=scale, size=(num_users, K))
    Q = np.random.normal(loc=0.0, scale=scale, size=(num_movies, K))

    # -------------------------------------------------------------------------
    # 5. Perform Pure SGD Algorithm
    # -------------------------------------------------------------------------
    print(f"\n======================================================================")
    print(f"RUNNING MODEL | K={K} | Alpha={alpha} | Max Epochs={epochs}")
    print(f"======================================================================")

    best_test_rmse = float('inf')

    for epoch in range(1, epochs + 1):
        np.random.shuffle(train_data)
        
        for u, m, r in train_data:
            err = r - np.dot(P[u], Q[m])
            P_u_old = P[u].copy()
            
            P[u] += alpha * err * Q[m]
            Q[m] += alpha * err * P_u_old

        if epoch % 5 == 0 or epoch == 1:
            train_rmse = compute_rmse(train_data, P, Q)
            test_rmse = compute_rmse(test_data, P, Q)
            print(f"Epoch {epoch:02d}/{epochs} | Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")

            if test_rmse < best_test_rmse:
                best_test_rmse = test_rmse
            elif test_rmse > best_test_rmse + 0.005 and epoch > 10:
                print(f"--> Test error bottomed out at epoch {epoch}. Stopping early.")
                break

    # -------------------------------------------------------------------------
    # 6. Generate Top 5 Recommendations & Bottom 5 Dislikes
    # -------------------------------------------------------------------------
    my_predictions = np.dot(P[MY_USER_IDX], Q.T)
    seen_movie_indices = {m_idx for u_idx, m_idx, r in ratings_tuples if u_idx == MY_USER_IDX}

    unseen_predictions = [
        (m_idx, my_predictions[m_idx]) 
        for m_idx in range(num_movies) if m_idx not in seen_movie_indices
    ]
    unseen_predictions.sort(key=lambda item: item[1], reverse=True)

    print(f"\nTOP 5 RECOMMENDED PICKS:")
    for rank, (m_idx, score) in enumerate(unseen_predictions[:5], start=1):
        raw_id = idx2movie[m_idx]
        print(f" {rank}. {id_to_title.get(raw_id, 'Unknown Title')}")

    print(f"\nBOTTOM 5 DISLIKES:")
    for rank, (m_idx, score) in enumerate(unseen_predictions[-5:], start=1):
        raw_id = idx2movie[m_idx]
        print(f" {rank}. {id_to_title.get(raw_id, 'Unknown Title')}")

    return P, Q, best_test_rmse

We run the following experiments below, detailed by the comments

In [40]:
# Experiment 1: Slower Learning Rate(K=20, alpha=0.005)
P1, Q1, rmse1 = run_matrix_completion_pipeline(df_filtered, my_raw_ratings, K=20, alpha=0.005, epochs=35)

# Experiment 2: Lower Rank (K=5) - Lower complexity, less capacity to memorize noise
P2, Q2, rmse2 = run_matrix_completion_pipeline(df_filtered, my_raw_ratings, K=5, alpha=0.05, epochs=35)

# Experiment 3: Higher Rank (K=50) and Faster Learning Rate (alpha=0.01)
P3, Q3, rmse3 = run_matrix_completion_pipeline(df_filtered, my_raw_ratings, K=50, alpha=0.01, epochs=35)


RUNNING MODEL | K=20 | Alpha=0.005 | Max Epochs=35
Epoch 01/35 | Train RMSE: 3.5789 | Test RMSE: 3.6914
Epoch 05/35 | Train RMSE: 1.0652 | Test RMSE: 1.2932
Epoch 10/35 | Train RMSE: 0.7928 | Test RMSE: 1.0173
Epoch 15/35 | Train RMSE: 0.7329 | Test RMSE: 0.9742
Epoch 20/35 | Train RMSE: 0.6973 | Test RMSE: 0.9638
Epoch 25/35 | Train RMSE: 0.6693 | Test RMSE: 0.9640
Epoch 30/35 | Train RMSE: 0.6452 | Test RMSE: 0.9682
Epoch 35/35 | Train RMSE: 0.6240 | Test RMSE: 0.9756
--> Test error bottomed out at epoch 35. Stopping early.

TOP 5 RECOMMENDED PICKS:
 1. Hate (Haine, La) (1995)
 2. Battlestar Galactica (2003)
 3. Boiler Room (2000)
 4. Brazil (1985)
 5. Fear and Loathing in Las Vegas (1998)

BOTTOM 5 DISLIKES:
 1. Akira (1988)
 2. Children of the Corn (1984)
 3. Waiting... (2005)
 4. Volcano (1997)
 5. Candyman (1992)

RUNNING MODEL | K=5 | Alpha=0.05 | Max Epochs=35
Epoch 01/35 | Train RMSE: 0.9423 | Test RMSE: 1.0164
Epoch 05/35 | Train RMSE: 0.8487 | Test RMSE: 0.9469
Epoch 10/35 

Notes from the above experiments:  
  
Model 1 (K=20, alpha=0.005)  
1. steady learning, completed all 35 epochs due to smaller step size
2. Test RMSE started rising at the end, due to lack of regularization it was adding noise.
3. excellent alignment with darker comedy, intense movies
4. cleared out generally low-rated movies, but not as much correlation with the dislikes  
  
Model 2 (K=5, alpha=0.05)
1. stopped early at epoch 25
2. high learning rate caused epoch 1 RMSE to be much lower
3. achieved the lowest absolute Test RMSE
4. top recommendations were not very specific due to limited K (limited categories to profile)
5. the dislikes are very accurate, bottom movies including rom-coms.  
    
Model 3 (K=50, alpha=0.01)
1. stopped quickly at epoch 15 due to large step size
2. overfit too fast
3. the top suggestions accurately capture sci-fi and epic action
4. the bottom dislikes accurately capture cheesier humor

Conclusions:  
  Changing K has a tradeoff. A higher K allows the model to adapt to more complex tastes, but can overfit rapidly due to the lack of lambda regularization in the algorithm. Having K too low makes the suggestions too general, and not accurately representative of the user's complex tastes. The most optimal alpha seen here was 0.005, as 0.05 accelerated the path to the minimum (reached in fewer epochs), but overshot it. 0.005 smoothly landed at the minimum across epochs. This tells us that an ideal configuration for an unregularized algorithm like our implementation runs with a K near 20 and alpha near 0.005.